# Match Win Probability — CricketModel Training

Run this notebook on **Google Colab** for GPU-accelerated training.

**Workflow:**
1. Mount Google Drive (your `t20.csv` must be at `MyDrive/cricket/t20.csv`)
2. Clone the repo to get latest `hyperparams.json`
3. Train CricketModel
4. Copy the JSON metrics line back to update EPOCH results

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify t20.csv is accessible
import os
DATA_PATH = '/content/drive/MyDrive/cricket/t20.csv'
assert os.path.exists(DATA_PATH), f"Not found: {DATA_PATH} — upload t20.csv to MyDrive/cricket/"
print(f'Data found: {os.path.getsize(DATA_PATH)/1e6:.1f} MB')

## 2. Install dependencies

In [ ]:
!pip install -q torch pandas numpy scikit-learn pyyaml joblib

## 3. Clone repo (gets latest hyperparams.json)

In [ ]:
import os

REPO_DIR = '/content/claude_wpa'
if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/tvganesh/claude_wpa.git {REPO_DIR}

os.chdir(REPO_DIR)
!cat projects/match_win_probability/hyperparams.json

## 4. Check GPU

In [ ]:
import torch

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
    print('Device: MPS')
else:
    device = torch.device('cpu')
    print('Device: CPU — consider enabling GPU in Runtime > Change runtime type')

## 5. Run training

This calls `evaluate.py train` exactly as EPOCH does locally.
The last output line is a JSON dict with the metrics.

In [ ]:
import subprocess, sys, json

# Override DATA_PATH to point to Google Drive
# evaluate.py reads data from PROJECT_ROOT/output/t20.csv
# We symlink Drive data → expected path
os.makedirs('output', exist_ok=True)
if not os.path.exists('output/t20.csv'):
    os.symlink(DATA_PATH, 'output/t20.csv')
    print('Symlinked t20.csv')

result = subprocess.run(
    [sys.executable, 'projects/match_win_probability/evaluate.py', 'train'],
    capture_output=False,   # show epoch progress live
    text=True
)
print('\nExit code:', result.returncode)

## 6. View saved metrics

In [ ]:
import glob, json

# Find the latest train_results.json
results_files = sorted(glob.glob('projects/match_win_probability/runs/*/train_results.json'))
if results_files:
    with open(results_files[-1]) as f:
        results = json.load(f)
    print(f"Run dir: {results_files[-1]}")
    print(f"Epochs run:      {results['epochs_run']}")
    print(f"final_train_acc: {results['final_train_acc']:.4f}")
    print(f"final_val_acc:   {results['final_val_acc']:.4f}")
    print(f"final_train_loss:{results['final_train_loss']:.4f}")
    print(f"final_val_loss:  {results['final_val_loss']:.4f}")
    print(f"train_eval_gap:  {results['train_eval_gap']:.4f}")

## 7. Plot loss & accuracy curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_range = range(1, results['epochs_run'] + 1)

ax1.plot(epochs_range, results['train_losses'], label='train loss')
ax1.plot(epochs_range, results['val_losses'],   label='val loss')
ax1.set_title('Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(epochs_range, results['train_accs'], label='train acc')
ax2.plot(epochs_range, results['val_accs'],   label='val acc')
ax2.set_title('Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.suptitle(f"CricketModel — val_acc={results['final_val_acc']:.4f}")
plt.tight_layout()
plt.show()